# Live training demo: a CNN learning from Nonspeech7k

Shows the full deep-learning loop on a **small subset of the training partition** (no test clips): initialisation, forward pass, loss, backpropagation, optimizer step, parameter updates, validation. The real experiments (`train_cnn.py`, `ast_finetune.py`) use the same code at full scale.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
REPO = Path.cwd().parent if Path.cwd().name == "professor_demo" else Path.cwd()
sys.path.insert(0, str(REPO))
# data and results live next to the repository; override with environment variables if moved
DATA = Path(os.environ.get("NS7K_DATA", REPO.parent / "data"))
RESULTS = Path(os.environ.get("NS7K_RESULTS", REPO.parent / "results" / "v3"))
from common import index_audio
from features import load_mono
from dl_data import CLASSES
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CPU threads:", torch.get_num_threads())
audio = index_audio(DATA / "audio")
splits = pd.read_csv(RESULTS / "dl" / "splits.csv")
print(len(audio), "audio files indexed;", len(splits), "clips in the split manifest")

In [ ]:
import torch.nn.functional as F
from train_cnn import LogMelCNN, augment
from dl_metrics import metrics
torch.manual_seed(0)
M = np.load(RESULTS / "dl" / "logmel.npy", mmap_mode="r")
tr = splits.index[splits.source_disjoint == "train"].values
va = splits.index[splits.source_disjoint == "val"].values
rng = np.random.default_rng(0)
tr_s = np.sort(np.concatenate([rng.choice(tr[splits.y.values[tr] == c], 40, replace=False) for c in range(7)]))
va_s = np.sort(np.concatenate([rng.choice(va[splits.y.values[va] == c], 15, replace=False) for c in range(7)]))
mu = np.asarray(M[tr_s], np.float32).mean((0, 2), keepdims=True)[0]; sd = np.asarray(M[tr_s], np.float32).std((0, 2), keepdims=True)[0] + 1e-5
Xtr = torch.from_numpy((np.asarray(M[tr_s], np.float32) - mu) / sd); ytr = torch.tensor(splits.y.values[tr_s])
Xva = torch.from_numpy((np.asarray(M[va_s], np.float32) - mu) / sd); yva = torch.tensor(splits.y.values[va_s])
print("train subset", Xtr.shape, "| val subset", Xva.shape, "| test clips used: 0")

## 1. Initialise the network

In [ ]:
model = LogMelCNN(7, 16)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 2. One step by hand: forward, loss, backward, optimizer step

In [ ]:
xb, yb = Xtr[:16], ytr[:16]
w0 = model.head.weight.detach().clone(); c0 = model.features[0].net[0].weight.detach().clone()
logits = model(xb)                                  # forward propagation
loss = F.cross_entropy(logits, yb)                  # loss
opt.zero_grad(); loss.backward()                    # backpropagation
print("logits shape", tuple(logits.shape), "| loss", round(loss.item(), 4))
print("gradient norm, first conv:", model.features[0].net[0].weight.grad.norm().item())
print("gradient norm, head      :", model.head.weight.grad.norm().item())
opt.step()                                          # optimizer step
print("first-conv weights changed by", (model.features[0].net[0].weight - c0).norm().item())
print("head weights changed by      ", (model.head.weight - w0).norm().item())

## 3. Short training run with validation after every epoch

In [ ]:
hist = []; gen = torch.Generator().manual_seed(0)
t0 = time.time()
for ep in range(1, 9):
    model.train(); perm = torch.randperm(len(Xtr), generator=gen); tot = 0
    for i in range(0, len(perm), 16):
        b = perm[i:i+16]
        loss = F.cross_entropy(model(augment(Xtr[b], gen)), ytr[b])
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
    model.eval()
    with torch.no_grad():
        lv = model(Xva); vl = F.cross_entropy(lv, yva).item()
    m = metrics(yva.numpy(), lv.argmax(1).numpy(), 7)
    hist.append(dict(epoch=ep, train_loss=tot/len(Xtr), val_loss=vl, val_acc=m["accuracy"], val_macro_f1=m["macro_f1"]))
    print(f"epoch {ep} train_loss {tot/len(Xtr):.3f} val_loss {vl:.3f} val_acc {m['accuracy']:.3f} val_macroF1 {m['macro_f1']:.3f} ({time.time()-t0:.0f}s)")
h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set(title="loss", xlabel="epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.val_acc, label="val accuracy"); ax[1].plot(h.epoch, h.val_macro_f1, label="val macro-F1"); ax[1].set(title="validation", xlabel="epoch", ylim=(0, 1)); ax[1].legend()
plt.show()

A 280-clip, 8-epoch run is only a demonstration of the training loop; the full experiment's learning curves are in `results/v3/report/G1_G2_learning_curves_*.png`.

## 4. Evidence that the pretrained AST was actually fine-tuned
Gradient norms on pretrained transformer weights during training, and how far those weights moved.

In [ ]:
run = RESULTS / "runs" / "ast_ft_source_disjoint_standard_s0"
print(open(run / "train.log").read().splitlines()[-1])
display(pd.read_csv(run / "grad_norms.csv").T.head(12))
drift = pd.read_csv(run / "weight_drift.csv", index_col=0)
drift.sort_values("relative_change", ascending=False).head(10)